# EfficientAD on MVTec AD — free cloud GPU (step 1g)

Runs **the same code and config as the repository** (`scripts/train_efficientad.py`, `configs/vision/efficientad.yaml`) and writes one **run bundle** per category.
Download the bundles and import them locally with `uv run python scripts/import_run_bundle.py <bundle.zip>`; the import re-evaluates on CPU and must reproduce the image AUROC within 0.005.

**Approvals (blueprint Rev 2 §3.9) — nothing below downloads until `APPROVE_DOWNLOADS = True`:**

| Item | Source | Size | Licence |
| --- | --- | --- | --- |
| MVTec AD (your own copy, from Google Drive) | your Drive | ≈ 5.3 GB | CC BY-NC-SA 4.0 — keep the Drive folder private |
| ImageNette2 (`imagenette2.tgz`, penalty loss) | s3.amazonaws.com/fast-ai-imageclas | ≈ 1.5 GB (verify) | ImageNet terms |
| `efficientad_pretrained_weights.zip` (teacher) | github.com/open-edge-platform/anomalib releases | verify | Apache-2.0 |

Expected runtime (estimate, not measured): 70 000 steps of EfficientAD-S ≈ 1–2 h per category on a T4. Start with one category; free quotas are limited.


In [ ]:
# --- settings -----------------------------------------------------------------
REPO_URL = "https://github.com/<owner>/industrial-quality-ai.git"  # your repository
GIT_REF = "v0.1-step1"  # tag or commit: the bundle records it, import checks it
CATEGORIES = ["bottle"]  # one at a time on free GPUs
MVTEC_ARCHIVE = "/content/drive/MyDrive/datasets/mvtec_anomaly_detection.tar.xz"
OUT_DIR = "/content/drive/MyDrive/iqs_bundles"
MAX_STEPS = None  # None = config (70 000); e.g. 200 for a smoke run
APPROVE_DOWNLOADS = False  # set True only after approving the table above

In [ ]:
import os
import subprocess
import sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no GPU!")
from google.colab import drive  # on Kaggle: attach the dataset instead and skip this

drive.mount("/content/drive")

In [ ]:
%cd /content
!rm -rf repo && git clone --quiet {REPO_URL} repo && cd repo && git checkout --quiet {GIT_REF} && git log --oneline -1
%cd /content/repo
# Keep the runtime's CUDA torch; install the pinned libraries and our two packages without deps.
!pip install --quiet -r finetune/efficientad/requirements.txt
!pip install --quiet --no-deps -e packages/common -e packages/vision

In [ ]:
# MVTec AD from your Drive: copy, verify SHA-256 against configs/assets.yaml, safe extract.
!python scripts/download_dataset.py mvtec_ad --archive "{MVTEC_ARCHIVE}" --approve --root /content/repo

In [ ]:
assert APPROVE_DOWNLOADS, (
    "Approve the ImageNette and teacher-weight downloads first (see the table)."
)
os.makedirs(OUT_DIR, exist_ok=True)
for cat in CATEGORIES:
    extra = [] if MAX_STEPS is None else ["--max-steps", str(MAX_STEPS)]
    cmd = [
        sys.executable,
        "scripts/train_efficientad.py",
        "--category",
        cat,
        "--approve-downloads",
        "--out-dir",
        OUT_DIR,
        *extra,
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

Download the `efficientad_<category>.zip` files from `OUT_DIR` and run locally:

```bash
uv run python scripts/import_run_bundle.py ~/Downloads/efficientad_bottle.zip
```
